# Trusted Zone — Spark batch (`spark_trusted_zone.py`)

Same code as [`spark_trusted_zone.py`](../spark_trusted_zone.py). Spark reads metadata **directly from MinIO via S3A**:

- `s3a://<LANDING_ZONE_BUCKET>/metadata/observations.csv`
- `s3a://<TRUSTED_ZONE_BUCKET>/metadata/observations.csv` (anti-join)
- Hot-path Kafka JSON: `landing-zone/metadata/kafka-events/hot-path/` → `device`, `time_recorded/added`

Then: trim strings, require `uuid` and `audio_path`, dedupe on `uuid`, anti-join trusted UUIDs, write `metadata/pending_workset.json`.

**Jupyter:** host driver → Docker Spark + localhost MinIO (`docker compose up -d`).

**Docker CLI:** `docker compose run --rm trusted-spark-batch`

## Paths and environment

In [ ]:
from pathlib import Path
import os
import sys

_here = Path.cwd().resolve()
_root = None
for p in [_here, *_here.parents]:
    if (p / "docker-compose.yml").is_file() and (p / "orchestrate.py").is_file():
        _root = p
        break
if _root is None:
    raise RuntimeError("Repo root not found — open the notebook from the BDM-Cymatics tree")

PROJECT_ROOT = str(_root)
TRUSTED_ZONE_DIR = _root / "trusted_zone"

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
if str(TRUSTED_ZONE_DIR) not in sys.path:
    sys.path.insert(1, str(TRUSTED_ZONE_DIR))

from dotenv import load_dotenv
load_dotenv(_root / ".env", override=False)

os.environ["SPARK_ALLOW_HOST_DRIVER"] = "1"
os.environ["SPARK_MASTER"] = os.environ.get("SPARK_MASTER") or "spark://localhost:7077"
_s3a = (os.environ.get("SPARK_S3A_ENDPOINT") or "").strip()
if not _s3a or "host.docker.internal" in _s3a:
    os.environ["SPARK_S3A_ENDPOINT"] = "http://127.0.0.1:9000"

print("PROJECT_ROOT =", PROJECT_ROOT)

## Import batch logic

In [ ]:
import importlib

import spark_trusted_zone

importlib.reload(spark_trusted_zone)

from shared.minio_helpers import create_minio_client
from spark_trusted_zone import (
    check_driver_java,
    run_spark_trusted_zone,
    _print_pending_summary,
)

## PySpark driver Java (11 or 17)

In [ ]:
import subprocess

_driver_java = check_driver_java()
print("Java installs usable for PySpark (11 preferred over 17):")
for _entry in _driver_java["available"]:
    _tag = "← driver" if _entry["java_home"] == _driver_java["java_home"] else ""
    print(f"  Java {_entry['java_major']}: {_entry['java_home']} {_tag}")
print(f"\nJAVA_HOME={_driver_java['java_home']}")

_check = subprocess.run(
    ["docker", "compose", "ps", "--status", "running", "minio", "spark-master"],
    cwd=PROJECT_ROOT,
    capture_output=True,
    text=True,
)
if _check.returncode != 0:
    raise RuntimeError(
        "Start Docker first: docker compose up -d\n" + (_check.stderr or _check.stdout)
    )
print(_check.stdout)

## Run: pending landing rows

Same as `python trusted_zone/spark_trusted_zone.py` / `main()`.

In [ ]:
client = create_minio_client()
pending = run_spark_trusted_zone(client)
_print_pending_summary(pending)